In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os
from pathlib import Path

notebook_path = "/u/skarmakar1/version_check/llm_steering-main/sk"
sys.path.append(notebook_path)

In [3]:
import torch
import numpy as np

from inversion_utils import *
import pickle
from sklearn.model_selection import train_test_split

In [4]:
SEED = 0
# SEED = 1

torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
np.random.seed(SEED)

torch.backends.cudnn.benchmark = True 
torch.backends.cuda.matmul.allow_tf32 = True

LLM = namedtuple('LLM', ['language_model', 'tokenizer', 'processor', 'name', 'model_type'])

In [5]:
model_type = 'llama'
MODEL_VERSION='3.1'
MODEL_SIZE='8B'

# model_type = 'gemma'
# MODEL_VERSION='2'
# # MODEL_VERSION='3'
# # MODEL_SIZE='1B'
# MODEL_SIZE='9B'
# # MODEL_SIZE='12B'

# model_type = 'qwen'
# MODEL_VERSION='3'
# MODEL_SIZE='4B'
# # MODEL_SIZE='8B'

llm = select_llm(model_type, MODEL_VERSION=MODEL_VERSION, MODEL_SIZE=MODEL_SIZE)

Loading meta-llama/Meta-Llama-3.1-8B-Instruct


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [6]:
hidden_layers = list(range(-llm.language_model.config.num_hidden_layers+1, 0))
print(hidden_layers)

[-31, -30, -29, -28, -27, -26, -25, -24, -23, -22, -21, -20, -19, -18, -17, -16, -15, -14, -13, -12, -11, -10, -9, -8, -7, -6, -5, -4, -3, -2, -1]


In [8]:
# Llama and gemma

# coef = 0.7 # llama
coef = 0.75
# coef = 0.8


max_tokens = 200

prompts = ["How is the weather in San Diego?"]

# prompts = ["It seems the family next doors has fallen ill."] # ('sympathetic', 'unsympathetic', 'social')
# prompts = ["I am walking home alone at night, I am a little scared.",] # ('brave', 'cowardly', 'social')
# prompts = ["You are designing a new vehicle. Describe its size and weight."] # ('wide', 'narrow', 'physical')
# prompts = ["Is the following statement correct or not?"] # ('correct', 'incorrect', 'logic') - weird!!!
# prompts = ["Describe this sound."] # ('agonizing', 'soothing', 'texture')
# prompts = ["Describe the following event or entity. Subject: ."] # ('long-term', 'short-term', 'time')
# prompts = ["I am walking home alone at night, is it safe?"] # ('safe', 'dangerous', 'state') - almost
# prompts = ["What is weight of a football?"] # ('precise', 'inaccurate', 'complexity')
# prompts = ["Explain how a toaster works."] # "basic"

category = "social"


path = f'../all_gitignore/directions_adjectives_llama/{category}/'

print(f"Coef: {coef}")

# c1 = "affectionate"
# c1 = "agitated"
# c1 = "brave"
# c1 = "sympathetic"
c1 = "joyful"

c1_controller = load_controller(llm, c1, path=path)
orig_c1 = c1_controller.directions

out = test_concept_vector(c1_controller, concept=c1, prompts=prompts, coef=coef, max_tokens=max_tokens, orig=True)
# out = test_concept_vector(c1_controller, concept=c1, prompts=prompts, coef=coef, max_tokens=max_tokens, orig=False)

Coef: 0.75
Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 16
M_batch_size         : 2048
n_components         : 1

Using xRFM library for RFM-based steering vector extraction
Detector found

========================== No Control ==========================
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

<|eot_id|><|start_header_id|>user<|end_header_id|>

How is the weather in San Diego?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

San Diego is known for its mild and pleasant climate year-round.  As of my cut-off knowledge in 2023, the weather in San Diego is generally warm and sunny, with average temperatures ranging from the mid-60s to mid-70s Fahrenheit (18-24°C) throughout the year.


In [ ]:
# import torch
# import pandas as pd

# def logit_lens_analysis(model, tokenizer, steering_vector, k=10):
#     """
#     Projects a steering vector onto the vocabulary and prints top/bottom tokens.
    
#     Args:
#         model: The LLM (e.g., Llama 3.1)
#         tokenizer: The matching tokenizer
#         steering_vector: Tensor of shape (4096,)
#         k: Number of top/bottom tokens to show
#     """
#     # 1. Get the Unembedding Matrix (Weights of the LM Head)
#     # In Llama, this is usually model.lm_head.weight
#     W_U = model.lm_head.weight.detach() # Shape: (Vocab, Hidden)
    
#     # 2. Project the vector
#     # normalize vector for cleaner comparison (optional, but good practice)
#     v = steering_vector / steering_vector.norm()
    
#     # Calculate dot product (Logit Attributions)
#     # (Vocab, Hidden) @ (Hidden,) -> (Vocab,)
#     logits = torch.matmul(W_U, v.to(W_U.device).to(W_U.dtype))
    
#     # 3. Get Top and Bottom tokens
#     top_vals, top_indices = torch.topk(logits, k)
#     bottom_vals, bottom_indices = torch.topk(logits, k, largest=False)
    
#     # 4. Decode
#     print(f"--- Top {k} Tokens (Positive Direction) ---")
#     for val, idx in zip(top_vals, top_indices):
#         token = tokenizer.decode(idx.item())
#         print(f"Token: {token:<15} | Logit Contribution: {val:.4f}")
        
#     print(f"\n--- Bottom {k} Tokens (Negative Direction) ---")
#     for val, idx in zip(bottom_vals, bottom_indices):
#         token = tokenizer.decode(idx.item())
#         print(f"Token: {token:<15} | Logit Contribution: {val:.4f}")

# # Example Usage:
# # logit_lens_analysis(model, tokenizer, my_happy_vector)

In [23]:
import torch
import torch.nn.functional as F

def logit_lens_steering(model, tokenizer, steering_vector, k=10, layernorm=True):
    """
    Projects a steering vector onto the vocabulary.
    
    Args:
        model: The Llama model
        tokenizer: The tokenizer
        steering_vector: Tensor of shape (4096,)
        k: Number of top/bottom tokens to show
    """

    # ---------------------------------------------------------------------------------

    # W_U = model.lm_head.weight.detach() # Shape: [Vocab_Size, Hidden_Dim]
    
    # if layernorm:
    #     # 2. Normalize the steering vector (Optional but recommended for scale invariance)
    #     # We norm it so we are looking at pure direction alignment
    #     v_norm = torch.nn.functional.normalize(steering_vector, dim=0)
    # else:
    #     v_norm = steering_vector

    # # 3. Apply Logit Lens (Project vector onto vocab)
    # # Note: W_U is usually [Vocab, Hidden], so we do MatMul(v, W_U.T)
    # # But PyTorch Linear layer stores weight as [Out, In], so we can just matmul directly
    # logits = torch.matmul(W_U, v_norm)
    # ---------------------------------------------------------------------------------

    # 1. Get the Unembedding Matrix (Output Head) & Final Norm
    # Note: In Llama, the output head is usually `lm_head` 
    # and the final norm is `model.norm`
    lm_head = model.lm_head
    final_norm = model.model.norm
    
    # 2. Pre-process the vector
    # Steering vectors are directions, so their norm is arbitrary.
    # However, the LayerNorm expects variance. 
    # For pure direction analysis, we can skip LN or apply it. 
    # Applying LN is safer to match the distribution W_U expects.
    v = steering_vector.to(model.device).float()
    
    if layernorm:
        # Option A: Apply Final LayerNorm (Standard Logit Lens)
        v_projected = final_norm(v)
    
    else:
        # Option B: Raw Projection (sometimes better for pure difference vectors)
        v_projected = v 
    
    # 3. Project to Vocab
    # Shape: (Vocab_Size,)
    logits = lm_head(v_projected)
    # ---------------------------------------------------------------------------------



    # 4. Get Top and Bottom Tokens
    top_vals, top_indices = torch.topk(logits, k)
    bottom_vals, bottom_indices = torch.topk(logits, k, largest=False)
    
    print("--- Concepts Amplified (Top K) ---")
    for i in range(k):
        token = tokenizer.decode([top_indices[i]])
        print(f"{token.strip()} ({top_vals[i]:.2f})")

    # print("\n--- Concepts Suppressed (Bottom K) ---")
    # for i in range(k):
    #     token = tokenizer.decode([bottom_indices[i]])
    #     print(f"{token.strip()} ({bottom_vals[i]:.2f})")

# Usage:
# logit_lens_steering(model, tokenizer, my_rfm_vector_layer_15)

In [27]:
layer = -2

logit_lens_steering(llm.language_model, llm.tokenizer, orig_c1[layer].squeeze(), layernorm=False)

print("\n"+"*"*50)

logit_lens_steering(llm.language_model, llm.tokenizer, orig_c1[layer].squeeze(), layernorm=True)

--- Concepts Amplified (Top K) ---
b (0.10)
б (0.07)
ascript (0.07)
488 (0.06)
บ (0.06)
Pf (0.06)
ILON (0.06)
*b (0.06)
imbledon (0.06)
ucht (0.06)

**************************************************
--- Concepts Amplified (Top K) ---
b (15.14)
б (10.45)
ascript (10.23)
488 (9.40)
*b (8.85)
บ (8.84)
ucht (8.74)
Pf (8.69)
=b (8.58)
imbledon (8.55)


In [ ]:
# --- Concepts Amplified (Top K) ---
# b (0.10)
# б (0.07)
# ascript (0.07)
# 488 (0.06)
# บ (0.06)
# Pf (0.06)
# ILON (0.06)
# *b (0.06)
# imbledon (0.06)
# ucht (0.06)

# --- Concepts Amplified (Top K) ---
# b (15.14)
# б (10.45)
# ascript (10.23)
# 488 (9.40)
# *b (8.85)
# บ (8.84)
# ucht (8.74)
# Pf (8.69)
# =b (8.58)
# imbledon (8.55)